In [1]:
%run start.py

Root set to: /home/bdudas/obesity_challange


In [2]:
import torch
import numpy as np
from src.data.vae_data import get_loaders
from omegaconf import OmegaConf
from src.models.transformerVAE import TransformerVAEEncoder, TransformerVAEDecoder,Transfomer_latent_Classifier


from src.models.vae_trainers import StateTrainer_latent

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
z_dim = 128
traning_config = OmegaConf.load("configs/traning.yaml")
encoder_config = OmegaConf.load("configs/encoder.yaml")
encoder_config.z_dim = z_dim

decoder_config = OmegaConf.load("configs/decoder.yaml")
decoder_config.z_dim = z_dim

classifier_config = OmegaConf.load("configs/classifier_latent.yaml")
classifier_config.z_dim = z_dim
trainer_config = OmegaConf.load("configs/trainer.yaml")

In [4]:
train_loader,valloader, *_ = get_loaders(path = "",batch_size=128)

In [5]:
encoder = TransformerVAEEncoder(**encoder_config)
decoder = TransformerVAEDecoder(**decoder_config)
classifier = Transfomer_latent_Classifier(**classifier_config)

model = StateTrainer_latent(encoder,decoder,categorizer=classifier,**trainer_config)

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:210: Attribute 'reconLoss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['reconLoss'])`.


In [6]:
cpkt_path = "misc/best_runs/latent_reg/checkpoints/epoch=19-step=5520.ckpt"

state_dict = torch.load(cpkt_path,weights_only=False)
model.load_state_dict(state_dict['state_dict'])

<All keys matched successfully>

# Generate low dimensional data

In [7]:
batch = next(iter(train_loader))
Xs, gene, state = batch

In [8]:
mu, logvar = model.encoder(Xs)
logvar = torch.clamp(logvar, min=-6.0, max=2.0)
z = model.reparameterize(mu, logvar)
# Shape of Z : (Batch, z_dim)
#z = z / (z.norm(dim=1, keepdim=True) + 1e-6)
Xs_hat = model.decoder(z)

loss_kld = model.compute_kl_loss(mu, logvar)

In [9]:
loss_kld

tensor(602.7365, grad_fn=<MeanBackward0>)

# Checking

In [7]:
import torch.nn as nn

In [11]:
batch_iter = iter(valloader)

In [8]:
import torchmetrics
from torchmetrics.classification import MulticlassConfusionMatrix
from torchmetrics.wrappers import ClasswiseWrapper
from tqdm import tqdm

In [9]:
aucMetric = torchmetrics.AUROC(num_classes=4, average=None,task="multiclass")
confmat = MulticlassConfusionMatrix(num_classes=4)
classes = ['pre_adipo', 'adipo', 'lipo', 'other']
classwice_auc  = ClasswiseWrapper(aucMetric,labels=classes)

In [14]:
class_aucs = {class_name: [] for class_name in classes}
pbar = tqdm(valloader, total=len(valloader))
for batch in pbar:
    Xs, _, state = batch
    mu, logvar = model.encoder(Xs)
    logvar = torch.clamp(logvar, min=-6.0, max=2.0)
    z = model.reparameterize(mu, logvar)
    # Shape of Z : (Batch, z_dim)
    #z = z / (z.norm(dim=1, keepdim=True) + 1e-6)
    Xs_hat = model.decoder(z)

    loss_recon = nn.functional.mse_loss(Xs_hat, Xs, reduction=model.reduction)
    loss_kld = model.compute_kl_loss(mu, logvar)
    loss_kld = torch.clamp(loss_kld, max=model.KLD_MAX)
    beta = model.kl_weight()

    class_logits = model.categorizer(z)
    if state.shape == class_logits.shape:
        state_indices = torch.argmax(state, dim=1)
        # 2. If state is [Batch, 1] -> Squeeze to [Batch]
    elif state.ndim == 2 and state.shape[1] == 1:
        state_indices = state.squeeze(1)
    else:
        state_indices = state
    loss_class = nn.functional.cross_entropy(class_logits, state)
    auc_scores = classwice_auc(class_logits,state_indices)
    for aval, class_name in zip(auc_scores.values(), classes):
        class_aucs[class_name].append(aval.item())   
    pbar.set_postfix({classes[i]: f"{np.mean(class_aucs[classes[i]]):.4f}" for i in range(len(classes))})

 19%|█▉        | 26/138 [00:41<02:59,  1.60s/it, pre_adipo=0.9896, adipo=0.9625, lipo=0.9795, other=0.9817]


KeyboardInterrupt: 

# Checking new loaders

In [10]:
from src.data.perturbation_data import get_loaders
train_loader,valloader, *_ = get_loaders(path = "",batch_size=128)

In [11]:
batch = next(iter(train_loader))

In [12]:
Xs, x_in,target_gene, state, input_state = batch


In [13]:
mu, logvar = model.encoder(x_in)
logvar = torch.clamp(logvar, min=-6.0, max=2.0)
z = model.reparameterize(mu, logvar)
Xs_hat = model.decoder(z)

loss_recon = nn.functional.mse_loss(Xs_hat, Xs, reduction=model.reduction)
loss_kld = model.compute_kl_loss(mu, logvar)


In [14]:
Xs_hat.shape

torch.Size([128, 32, 675])

In [15]:
Xs_hat[0] - Xs_hat[2]

tensor([[-0.6326,  0.3386,  0.0425,  ...,  0.2876, -0.3029, -0.5280],
        [ 0.0985, -0.1597,  0.3192,  ..., -0.1666, -0.0787, -0.0116],
        [-0.1069,  0.5785,  0.0783,  ...,  0.3342,  0.4352,  0.0941],
        ...,
        [ 0.1259,  0.1356, -0.1183,  ...,  0.0892, -0.3755, -0.6836],
        [ 0.6096, -0.2319, -0.1511,  ...,  0.1083, -0.7117,  0.2933],
        [-0.2575,  0.0652,  0.2829,  ..., -0.0696,  0.3562, -0.5535]],
       grad_fn=<SubBackward0>)

In [16]:
torch.nn.functional.cosine_similarity(Xs_hat[0].flatten(), Xs_hat[2].flatten(), dim=0)

tensor(0.9767, grad_fn=<SumBackward1>)

In [17]:
torch.nn.functional.cosine_similarity(Xs_hat[0].flatten(), Xs_hat[1].flatten(), dim=0)

tensor(0.9830, grad_fn=<SumBackward1>)

In [18]:
Xs_comp = Xs_hat[input_state[:,1] == 1.]
Xs_rest = Xs_hat[input_state[:,1] == 0.]

In [19]:
inside_similarities = []
outside_similarities = []
for xs in Xs_comp:
    for xr in Xs_rest:
        outside_similarities.append(torch.nn.functional.cosine_similarity(xs.flatten(), xr.flatten(), dim=0).item())
    for xs2 in Xs_comp:
        if not torch.equal(xs,xs2):
            inside_similarities.append(torch.nn.functional.cosine_similarity(xs.flatten(), xs2.flatten(), dim=0).item())

In [21]:
np.mean(inside_similarities), np.mean(outside_similarities)

(0.9831380364424258, 0.9788014274873074)

In [24]:
for idx in range(4):

    Xs_comp = Xs_hat[input_state[:,idx] == 1.]
    Xs_rest = Xs_hat[input_state[:,idx] == 0.]

    inside_similarities = []
    outside_similarities = []
    for xs in Xs_comp:
        for xr in Xs_rest:
            outside_similarities.append(torch.nn.functional.cosine_similarity(xs.flatten(), xr.flatten(), dim=0).item())
        for xs2 in Xs_comp:
            if not torch.equal(xs,xs2):
                inside_similarities.append(torch.nn.functional.cosine_similarity(xs.flatten(), xs2.flatten(), dim=0).item())

    print("Inner similarities:",np.mean(inside_similarities),"Outer similarities:", np.mean(outside_similarities))

Inner similarities: 0.9711356229252286 Outer similarities: 0.9707407037080344
Inner similarities: 0.9831380364424258 Outer similarities: 0.9788014274873074
Inner similarities: 0.9816255104089644 Outer similarities: 0.9784334990525453
Inner similarities: 0.9764257414000375 Outer similarities: 0.9749476887711457
